In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [2]:
def make_cv_folds(train_pool, dates, n_splits=5):
    for fold, (train_idx, val_idx) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(dates)):
        train_dates = dates[train_idx]
        val_dates = dates[val_idx]
        assert not set(train_dates) & set(val_dates)
        train_mask = train_pool['FlightDate'].isin(train_dates)
        val_mask = train_pool['FlightDate'].isin(val_dates)
        train_fold = train_pool[train_mask]
        val_fold = train_pool[val_mask]
        yield fold, train_fold, val_fold, train_dates, val_dates

In [3]:
df = pd.read_csv('../data/interim/seattle_ontime_clean.csv')
df.shape
df.columns


Index(['Unnamed: 0', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline',
       'IATA_CODE_Reporting_Airline',
       ...
       'Div4TailNum', 'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID',
       'Div5WheelsOn', 'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff',
       'Div5TailNum', 'Unnamed: 109'],
      dtype='object', length=111)

In [4]:

df['FlightDate'] = pd.to_datetime(df["FlightDate"])
df['FlightDate'].head()

0   2024-08-01
1   2024-08-02
2   2024-08-03
3   2024-08-04
4   2024-08-01
Name: FlightDate, dtype: datetime64[ns]

In [5]:
post_flight = [
    'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups',
    'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'ArrTime', 'ArrDelayMinutes',
    'ArrDel15', 'ArrivalDelayGroups', 'ActualElapsedTime', 'AirTime',
    'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay',
    'FirstDepTime', 'TotalAddGTime', 'LongestAddGTime',
    'Cancelled', 'CancellationCode', 'Diverted',
    'DivAirportLandings', 'DivReachedDest', 'DivActualElapsedTime', 'DivArrDelay',
    'DivDistance',
    'Div1Airport', 'Div1AirportID', 'Div1AirportSeqID', 'Div1WheelsOn',
    'Div1TotalGTime', 'Div1LongestGTime', 'Div1WheelsOff', 'Div1TailNum',
    'Div2Airport', 'Div2AirportID', 'Div2AirportSeqID', 'Div2WheelsOn',
    'Div2TotalGTime', 'Div2LongestGTime', 'Div2WheelsOff', 'Div2TailNum',
    'Div3Airport', 'Div3AirportID', 'Div3AirportSeqID', 'Div3WheelsOn',
    'Div3TotalGTime', 'Div3LongestGTime', 'Div3WheelsOff', 'Div3TailNum',
    'Div4Airport', 'Div4AirportID', 'Div4AirportSeqID', 'Div4WheelsOn',
    'Div4TotalGTime', 'Div4LongestGTime', 'Div4WheelsOff', 'Div4TailNum',
    'Div5Airport', 'Div5AirportID', 'Div5AirportSeqID', 'Div5WheelsOn',
    'Div5TotalGTime', 'Div5LongestGTime', 'Div5WheelsOff', 'Div5TailNum',
]
drop_cols = [
    'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'Tail_Number',
    'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID',
    'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
    'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID',
    'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac',
    'DepTimeBlk', 'ArrTimeBlk', 'Flights', 'DistanceGroup','Origin','Unnamed: 0', 'Unnamed: 109'
]


df = df.drop(columns=post_flight + drop_cols)
df.columns

Index(['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate',
       'IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline',
       'Dest', 'CRSDepTime', 'CRSArrTime', 'ArrDelay', 'CRSElapsedTime',
       'Distance'],
      dtype='object')

Post-flight columns like DepDelay, TaxiOut and ActualElapsedTime do not exist when someone is booking a flight. Training on them gives a model that scores well and cannot be deployed, so they are dropped here.

The columns in drop_cols are duplicate encodings of information that other kept columns already carry. Origin is dropped for a different reason, which is that every row is SEA after filtering, so the column is constant and there is no pattern in it to find.

The full reasoning is in `README.md`, "Modeling Guardrails".

In [6]:
cutoff = pd.Timestamp('2025-10-01')  # start of Q4; leaves 88% for training, 12% held out
df['DepHour'] = df['CRSDepTime'] // 100
train_pool = df[df['FlightDate'] < cutoff]
test = df[df['FlightDate'] >= cutoff]

dates = np.sort(train_pool['FlightDate'].unique())  # split on these, not on rows: flight counts per day vary 242-555
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    print(fold, train_fold.shape[0], val_fold.shape[0], train_dates.min(), train_dates.max(), val_dates.min(), val_dates.max())


print(len(train_pool), train_pool['FlightDate'].nunique(), train_pool['FlightDate'].min(), train_pool['FlightDate'].max())
print(len(test), test['FlightDate'].nunique(), test['FlightDate'].min(), test['FlightDate'].max())
assert train_pool['FlightDate'].max() < test['FlightDate'].min()
assert len(train_pool) + len(test) == len(df)


0 41901 51554 2024-01-01T00:00:00.000000000 2024-04-18T00:00:00.000000000 2024-04-19T00:00:00.000000000 2024-08-02T00:00:00.000000000
1 93455 49684 2024-01-01T00:00:00.000000000 2024-08-02T00:00:00.000000000 2024-08-03T00:00:00.000000000 2024-11-16T00:00:00.000000000
2 143139 41431 2024-01-01T00:00:00.000000000 2024-11-16T00:00:00.000000000 2024-11-17T00:00:00.000000000 2025-03-02T00:00:00.000000000
3 184570 47184 2024-01-01T00:00:00.000000000 2025-03-02T00:00:00.000000000 2025-03-03T00:00:00.000000000 2025-06-16T00:00:00.000000000
4 231754 53895 2024-01-01T00:00:00.000000000 2025-06-16T00:00:00.000000000 2025-06-17T00:00:00.000000000 2025-09-30T00:00:00.000000000
285649 639 2024-01-01 00:00:00 2025-09-30 00:00:00
38841 92 2025-10-01 00:00:00 2025-12-31 00:00:00


The cutoff is 2025-10-01. Everything before it is the train pool, which is 639 dates and 285,649 rows. Everything on or after it is test, which is 92 dates and 38,841 rows, and it is not touched until Issue 10. There is no separate validation split, so every candidate model is scored on these same 5 folds.

TimeSeriesSplit is run over the array of unique dates instead of over rows. Flight rows are not equally spaced, since there are 242 to 555 flights per day, while calendar dates are. gap=0 is the default and it is kept deliberately, because no feature in this notebook is lagged or rolling.

More detail in `README.md`, "Evaluation Strategy".

In [7]:
profile_cols = ['IATA_CODE_Reporting_Airline', 'Flight_Number_Reporting_Airline', 'Dest']
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    profile_median = train_fold.groupby(profile_cols)['ArrDelay'].median()
    val_fold = val_fold.merge(profile_median.rename('pred').reset_index(), on=profile_cols, how='left')  # left join: no matching profile leaves pred as NaN
    print(fold, val_fold['pred'].isna().sum(), len(val_fold))  # unmatched count vs fold size

    # Fold 0: 32% of validation rows have no matching profile, and that understates it
    # because profiles with 1-2 rows still get a median. ArrDelay has a fat right tail
    # (max 3359), so one outlier swings a median that small. Hence the n>=10 threshold below.


    

0 16924 51554
1 9959 49684
2 6072 41431
3 11989 47184
4 5033 53895


Fold 0 showed 32% of validation rows with no matching profile in that fold's training rows. The real number is higher than that, because a median is currently computed for every profile that appears even once, so a profile with 1 or 2 rows counts as a match instead of counting as a miss.

The threshold checks two things per row. Whether the profile exists in the training data at all, and whether it has at least 10 rows to compute a median from.

More detail in `README.md`, "Naive Baseline".

n=10 was picked by checking fold 0, the thinnest fold, since that is where the choice matters most. At n=10, 39.9% of individual profiles fall below the threshold but only 5.0% of that fold's training rows sit in them, because thin profiles do not carry much row weight.

n=5 excludes only 1.2% of rows but trusts a median computed from as few as 5 points, which is risky given ArrDelay's right tail. n=15 excludes 7.7% and n=20 excludes 10.2%, so both cost data without buying stability. The same threshold is used at every rung and kept fixed across all 5 folds.

More detail in `README.md`, "Naive Baseline".

In [8]:
# Thresholding approach
mae_scores =[]  # MAE per fold, full ladder
ladder_mae=[]  # MAE on fallback rows only, so rung 1 does not dilute the comparison
flat_mae =[]  # same rows, but sent straight to rung 3; the gap is what rung 2 earns
for fold, train_fold, val_fold, train_dates, val_dates in make_cv_folds(train_pool, dates):
    profile_stats = train_fold.groupby(profile_cols)['ArrDelay'].agg(['median', 'count'])
    reliable_profiles = profile_stats[profile_stats['count'] >= 10]['median']  # n>=10 only
    val_fold = val_fold.merge(reliable_profiles.rename('pred').reset_index(), on=profile_cols, how='left')
    val_fold['rung'] = np.where(val_fold['pred'].notna(),1,np.nan)  # track which rung answered, so the baseline cannot be a global median in disguise
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    # fallback rate rises once thin profiles are excluded, as expected

    coarse_stats = train_fold.groupby(['IATA_CODE_Reporting_Airline', 'DepHour'])['ArrDelay'].agg(['median', 'count'])
    reliable_coarse = coarse_stats[coarse_stats['count'] >= 10]['median']  # same n>=10 bar at every rung
    val_fold = val_fold.merge(reliable_coarse.rename('pred_coarse').reset_index(), on=['IATA_CODE_Reporting_Airline', 'DepHour'], how='left')
    unresolved = val_fold['pred'].isna()  # snapshot before rung 2 fills anything
    val_fold['pred'] = val_fold['pred'].fillna(val_fold['pred_coarse'])
    val_fold.loc[unresolved & val_fold['pred'].notna(),'rung'] = 2  # rung 2 filled only what rung 1 left empty, so the AND isolates exactly those
    # print(fold, val_fold['pred'].isna().sum(), len(val_fold))

    still_unresolved = val_fold['pred'].isna()
    global_median = train_fold['ArrDelay'].median()
    val_fold['pred'] = val_fold['pred'].fillna(global_median)
    val_fold.loc[still_unresolved, 'rung'] = 3
    # print(fold,val_fold['pred'].isna().sum())
    non_rung_1 = val_fold[val_fold['rung'] != 1]  # fallback rows only
    fold_ladder_mae = (non_rung_1['ArrDelay'] - non_rung_1['pred']).abs().mean()
    ladder_mae.append(fold_ladder_mae)
    
    fold_flat_mae = (non_rung_1['ArrDelay']- global_median).abs().mean()
    flat_mae.append(fold_flat_mae) 
    fold_mae = (val_fold['ArrDelay'] - val_fold['pred']).abs().mean()
    mae_scores.append(fold_mae)
    print(fold, val_fold['rung'].value_counts().sort_index().to_dict())  # rung usage per fold

print(np.mean(mae_scores), np.std(mae_scores))  # 19.95 min mean, 1.21 std

# Low std means performance holds across seasons.

print(np.mean(ladder_mae),np.std(ladder_mae))
print(np.mean(flat_mae),np.std(flat_mae))
print(np.mean(flat_mae) - np.mean(ladder_mae))  # rung 2's lift: 0.415 min for carrier+DepHour, vs 0.149 for carrier+Dest

0 {1.0: 30283, 2.0: 20641, 3.0: 630}
1 {1.0: 39274, 2.0: 10143, 3.0: 267}
2 {1.0: 34109, 2.0: 7308, 3.0: 14}
3 {1.0: 34643, 2.0: 12371, 3.0: 170}
4 {1.0: 39394, 2.0: 14356, 3.0: 145}
19.951910939426405 1.209756490038425
20.42569170229878 1.433145591603604
20.84019793146333 1.3210408121534298
0.41450622916454805


The first rung 2 grouped by carrier and destination and lifted 0.149 minutes over falling straight to the flat global median, which is close to nothing. 11 candidate groupings were then scored on the same 5 folds instead of guessing a replacement, and carrier with departure hour won at 0.415 minutes.

Destination actively hurts. Carrier alone lifts 0.210, which beats carrier and destination at 0.149, so destination is not adding signal, it is splitting groups into smaller and noisier ones. Departure hour carries a real effect that destination does not, since median ArrDelay by scheduled departure hour swings about 9 to 10 minutes across the day.

A ceiling check puts the total headroom available to any median based rung at about 1.70 minutes, so a model that does not beat this baseline by much is not necessarily broken.

More detail in `README.md`, "What Went Wrong".

In [9]:
df['dep_minutes'] = (df['CRSDepTime'] // 100) * 60 + (df['CRSDepTime'] % 100)  # HHMM is not a quantity: 10:59 to 11:00 is a 41 unit jump raw, +1 in minutes
df['arr_minutes'] = (df['CRSArrTime'] // 100) * 60 + (df['CRSArrTime'] % 100)
df[['CRSDepTime', 'dep_minutes', 'CRSArrTime', 'arr_minutes']].describe()  # min of 1 is 00:01, not a bad value

df['dep_sin'] = np.sin(2 * np.pi * df['dep_minutes'] / 1440)  # minutes still break at midnight: 1439 and 1 are 2 min apart, 1438 apart as numbers.
# Mapping to a circle fixes it: 23:59 and 00:01 land 0.0087 apart in sin/cos space.
df['dep_cos'] = np.cos( 2 * np.pi * df['dep_minutes'] / 1440)  # cos too, or two different times share one sin value

df['arr_sin'] = np.sin(2 * np.pi * df['arr_minutes'] / 1440)
df['arr_cos'] = np.cos(2 * np.pi * df['arr_minutes'] / 1440)

CRSDepTime and CRSArrTime are clock readings in HHMM, not quantities. 10:59 to 11:00 is one minute of real time but a 41 unit jump as an integer. Converting to minutes since midnight makes that same step a clean +1.

That conversion does not fix the day boundary. 23:59 becomes 1439 and 00:01 becomes 1, which is 2 minutes apart in real time and 1438 apart as numbers. Putting the angle through sin and cos places each time on a circle, where those two land 0.0087 apart. Both functions are needed, because sin alone maps 6am and 6pm to the same value.

Month and DayOfWeek wrap around too, but at 12 and 7 levels one-hot handles it, so periodic encoding is only worth it for the 1,440 value time columns.

In [10]:
categorical_cols = ['IATA_CODE_Reporting_Airline', 'Dest', 'Month', 'DayOfWeek']
ohe = OneHotEncoder(drop='first')

drop="first" is used for the linear model. One-hot encoding the 11 carriers gives 11 dummy columns that sum to 1 on every row, and the intercept is a coefficient on a hidden column that is always 1, so the intercept column and the sum of the dummies are identical row for row. Any amount can be subtracted from the intercept and added to every dummy coefficient without changing a single prediction, so there is no unique answer and a number like "carrier AS adds 5 minutes" means nothing.

drop="first" removes one dummy column. That category becomes all zeros and has nothing left to absorb a compensating shift.

In [11]:
open_cols = ['IATA_CODE_Reporting_Airline','Dest']  # open sets: cannot enumerate every carrier or destination ahead of time
closed_cols = ['Month', 'DayOfWeek']  # closed sets: 1-12 and 1-7 are known, so unknowns are impossible

ohe_open = OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist',min_frequency=4)


In [12]:
ohe_closed = OneHotEncoder(drop='first', categories=[list(range(1, 13)), list(range(1, 8))]) 

Two encoders in the ColumnTransformer, split by whether the category set is closed rather than by cardinality. Cardinality misleads here, because Month has 12 values and fails while carrier has 11 and does not.

Month and DayOfWeek can be written out in advance, so they get an explicit categories= of 1-12 and 1-7 and unknown becomes impossible. This is the column that actually breaks. The folds are temporal, so fold 0 trains on January to April and May through August arrive in its validation set as categories the encoder has never seen, in 3 of 5 folds. No min_frequency value fixes that, because those months are not rare, they are absent.

Dest and carrier cannot be enumerated ahead of time, since an airline can add a route at SEA, so they keep min_frequency=4 and handle_unknown="infrequent_if_exist". min_frequency=2 was tried first and built no bucket at all, because the thinnest Dest in folds 0 and 1 has 3 rows.

More detail in `README.md`, "What Went Wrong".

In [13]:
numeric_cols = ['DayofMonth', 'CRSElapsedTime', 'Distance', 'dep_sin', 'dep_cos', 'arr_sin', 'arr_cos']

Year, Quarter, FlightDate and Flight_Number_Reporting_Airline are all still in df and none of them are handed to the ColumnTransformer. Each is left out for a different reason.

Year takes two values here, 2024 and 2025, and the app predicts 2026 onward, so every prediction it will ever make is for a year the model has never seen. Quarter is a deterministic function of Month, so its dummy columns can be reconstructed exactly from Month's, which is the same collinearity that drop="first" exists to fix. FlightDate never recurs, so there is nothing in it to generalise from, and it stays only because it is the split key and the Issue 3 grouping key. Flight_Number_Reporting_Airline has 2,383 distinct values on the train pool against about 120 columns for all four categoricals that were kept, and it stays as the profile key and as the app's user input in Issue 14.

Keeping a column because it is useful for splitting, grouping or the app is a fine reason to keep a column. It is not a reason to feed it to a model.

More detail in `README.md`, "Modeling Guardrails".

In [14]:
preprocessor = ColumnTransformer([
    ('open', ohe_open, open_cols),
    ('closed', ohe_closed, closed_cols),
    ('num', 'passthrough', numeric_cols),
])

An encoder one-hot encodes every column it is given, so handing it the whole dataframe would treat each distinct Distance value as its own category. Distance has 100 distinct values and CRSElapsedTime has 364, so those two alone would come back as about 460 junk columns. ColumnTransformer routes specific columns to specific transformers. It is built from (name, transformer, columns) triples, and the numerics get the string "passthrough", which copies them through untouched.

What decides where a column goes is not whether it is a number. Month and DayOfWeek are integers and still go to a one-hot encoder. The test is whether the number is a label or a quantity. Distance 2400 really is twice 1200, but Month 12 is not twice Month 6.

The output width is not identical in every fold. Fold 0's 41,901 rows contain 85 destinations and fold 4's 231,754 rows contain 93, because each fold's encoder learns its category list from the rows it is fitted on. If fold 0 knew about all 93 it would mean destinations from 2025 had leaked backwards into a model that is only supposed to know early 2024. This is also why the preprocessor has to sit inside a Pipeline instead of being fitted once on the whole train pool and sliced into folds afterward.

More detail in `README.md`, "Evaluation Strategy".

In [15]:
fold0_train = next(make_cv_folds(train_pool,dates))[1]  # thinnest fold, 41,901 rows, where min_frequency=2 had failed
ohe_open.fit(fold0_train[open_cols])
print(ohe_open.infrequent_categories_)
unseen = ohe_open.transform(pd.DataFrame({'IATA_CODE_Reporting_Airline': ['ZZ'], 'Dest': ['ZZZ']})).toarray()  # unseen values must land in the infrequent column, not all-zeros like the dropped reference
print(unseen)
print(ohe_open.get_feature_names_out())

[None, array(['HDN'], dtype=object)]
[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]]
['IATA_CODE_Reporting_Airline_AS' 'IATA_CODE_Reporting_Airline_B6'
 'IATA_CODE_Reporting_Airline_DL' 'IATA_CODE_Reporting_Airline_F9'
 'IATA_CODE_Reporting_Airline_HA' 'IATA_CODE_Reporting_Airline_MQ'
 'IATA_CODE_Reporting_Airline_NK' 'IATA_CODE_Reporting_Airline_OO'
 'IATA_CODE_Reporting_Airline_UA' 'IATA_CODE_Reporting_Airline_WN'
 'Dest_ALW' 'Dest_ANC' 'Dest_ATL' 'Dest_AUS' 'Dest_BLI' 'Dest_BNA'
 'Dest_BOI' 'Dest_BOS' 'Dest_BUR' 'Dest_BWI' 'Dest_BZN' 'Dest_CHS'
 'Dest_CLE' 'Dest_CLT' 'Dest_CMH' 'Dest_CVG' 'Dest_DAL' 'Dest_DCA'
 'Dest_DEN' 'Dest_DFW' 'Dest_DTW' 'Dest_EUG' 'Dest_EWR' 'Dest_FAI'
 'Dest_FAT' 'Dest_FCA' 'Dest_FLL' 'Dest_GEG' 'Dest_GTF' 'Dest_HLN'
 

/opt/anaconda3/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


The encoder was run on fold 0, which is the thinnest fold at 41,901 training rows and the exact case where min_frequency=2 had failed. Folds 2 through 4 each have a destination with exactly 1 row, so any of those would have built a bucket anyway and hidden the problem.

infrequent_categories_ came back as [None, array(['HDN'])]. None for carrier means no bucket was built, which is enough to know that an unseen carrier comes out all zeros before transforming anything at all.

Transforming a row unseen in both columns returned a 94 wide vector with a single 1 at Dest_infrequent_sklearn. Dest worked. Carrier is all zeros, and the carrier feature names list only 10 of the 11 with AA missing, so an unseen carrier encodes identically to American Airlines. No encoder setting fixes this honestly, so it is handled as input validation in Issue 14 instead of being distorted into the encoder here.

More detail in `README.md`, "What Went Wrong".